# ComfyUI Model Downloader (Incremental)

Dedicated Colab session to incrementally download and verify ComfyUI models, workflow templates, and input assets directly into `/content/drive/MyDrive/colab-llm/comfyui/`.

Run it with the service script (`services/<name>.py up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "comfyui-models")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8189"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not os.path.isdir("/content/drive/MyDrive"):                 # Drive not mounted -> local fallback
    DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    logf = open(f"{WORK}/{name}.log", "a")
    return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                            env={**os.environ, **(env or {})}, start_new_session=True)

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        os.chmod(path, os.stat(path).st_mode | stat.S_IEXEC)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- incremental ComfyUI model downloader (saves directly to /content/drive/MyDrive/colab-llm/comfyui)
import hashlib
from pathlib import Path

COMFY_DATA = Path(os.environ.get("CT_COMFY_DATA_DIR", "/content/drive/MyDrive/colab-llm/comfyui"))
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted. Run via `./services/comfyui-models.py up`.")
COMFY_DATA.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"], check=True)
from huggingface_hub import HfApi, hf_hub_download

# Built-in model packs (each item: repo / filename / dest_rel OR url / dest_rel)
PACKS = {
    "minimax-h3-native": [
        # Diffusion models (T2V / I2V / FL2VA + R2V / Ref2VA)
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
         "dest": "models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors"},
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
         "dest": "models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"},
        # Text encoder
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
         "dest": "models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"},
        # Video & Audio VAEs
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "vae/minimax_h3_video_vae_int8_convrot.safetensors",
         "dest": "models/vae/minimax_h3_video_vae_int8_convrot.safetensors"},
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "vae/minimax_h3_audio_vae_fp32.safetensors",
         "dest": "models/vae/minimax_h3_audio_vae_fp32.safetensors"},
        # Lightning Turbo LoRAs (8-step for FL2V, 4-step for Ref2V)
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
         "dest": "models/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"},
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
         "dest": "models/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"},
        # Embeddings
        {"repo": "Comfy-Org/MiniMax-H3",
         "filename": "embeddings/minimaxh3_art_is_explosion.safetensors",
         "dest": "models/embeddings/minimaxh3_art_is_explosion.safetensors"},
        # Official workflow templates -> user/default/workflows/
        {"url": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json",
         "dest": "user/default/workflows/video_minimax_h3_t2v.json"},
        {"url": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_i2v.json",
         "dest": "user/default/workflows/video_minimax_h3_i2v.json"},
        {"url": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_r2v.json",
         "dest": "user/default/workflows/video_minimax_h3_r2v.json"},
        # Official example input images -> input/
        {"url": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/input/transparent_rgb_gaming_mouse.png",
         "dest": "input/transparent_rgb_gaming_mouse.png"},
        {"url": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/input/red_superboy_on_city_roof.png",
         "dest": "input/red_superboy_on_city_roof.png"},
        {"url": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/input/mecha_dragon_lightning.png",
         "dest": "input/mecha_dragon_lightning.png"},
    ],
}

MANIFEST_PATH = COMFY_DATA / "models" / ".download-manifest.json"
STATUS_PATH = Path(WORK) / "status.json"
STAGE_DIR = Path("/content/hf-stage")
STAGE_DIR.mkdir(parents=True, exist_ok=True)

def mlog(msg):
    print(time.strftime("[%F %T] ") + msg, flush=True)

def load_manifest():
    if MANIFEST_PATH.is_file():
        try:
            return json.loads(MANIFEST_PATH.read_text())
        except Exception:
            pass
    return {"files": {}}

def save_manifest(manifest):
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = MANIFEST_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n")
    tmp.replace(MANIFEST_PATH)

def write_status(state_dict):
    tmp = STATUS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(state_dict, indent=2, ensure_ascii=False) + "\n")
    tmp.replace(STATUS_PATH)

# Build incremental queue from CT_MODEL_PACKS + persistent queue file + CT_EXTRA_MODELS
active_packs = [p.strip() for p in os.environ.get("CT_MODEL_PACKS", "minimax-h3-native").split(",") if p.strip()]
items = []
for p in active_packs:
    if p not in PACKS:
        raise ValueError(f"Unknown model pack '{p}' (available: {list(PACKS)})")
    items.extend(PACKS[p])

queue_file = COMFY_DATA / "models" / "download-queue.json"
if queue_file.is_file():
    try:
        items.extend(json.loads(queue_file.read_text()))
    except Exception as e:
        mlog(f"Warning: could not parse {queue_file}: {e}")

extra_json = os.environ.get("CT_EXTRA_MODELS", "").strip()
if extra_json:
    items.extend(json.loads(extra_json))

# Resolve remote metadata (size & sha256) per HF repo once
repo_meta = {}
api = HfApi()
for it in items:
    repo = it.get("repo")
    if repo and repo not in repo_meta:
        info = api.model_info(repo, files_metadata=True)
        repo_meta[repo] = {
            s.rfilename: {"size": s.size, "sha256": (s.lfs or {}).get("sha256") if isinstance(s.lfs, dict) else getattr(s.lfs, "sha256", None)}
            for s in info.siblings
        }

manifest = load_manifest()
summary = {"packs": active_packs, "total": len(items), "completed": 0, "skipped": 0, "current": None, "files": []}
write_status(summary)

for idx, it in enumerate(items, 1):
    dest_rel = it["dest"]
    target = COMFY_DATA / dest_rel
    target.parent.mkdir(parents=True, exist_ok=True)
    rec = manifest["files"].get(dest_rel, {})

    if "repo" in it:
        repo, fn = it["repo"], it["filename"]
        meta = repo_meta.get(repo, {}).get(fn, {})
        exp_size = it.get("size") or meta.get("size")
        exp_sha = it.get("sha256") or meta.get("sha256")
        if target.is_file() and exp_size and target.stat().st_size == exp_size:
            if not exp_sha or rec.get("sha256") == exp_sha or rec.get("size") == exp_size:
                mlog(f"[{idx}/{len(items)}] SKIP (already on Drive, {exp_size/1e9:.2f} GB): {dest_rel}")
                manifest["files"][dest_rel] = {"repo": repo, "filename": fn, "size": exp_size, "sha256": exp_sha}
                save_manifest(manifest)
                summary["skipped"] += 1
                summary["completed"] += 1
                summary["files"].append({"dest": dest_rel, "size": exp_size, "status": "cached"})
                write_status(summary)
                continue

        summary["current"] = {"index": idx, "dest": dest_rel, "size": exp_size, "source": f"{repo}/{fn}"}
        write_status(summary)
        mlog(f"[{idx}/{len(items)}] DOWNLOADING ({ (exp_size or 0)/1e9:.2f} GB): {repo}/{fn} -> {dest_rel}")
        t0 = time.monotonic()
        staged = Path(hf_hub_download(repo_id=repo, filename=fn, local_dir=STAGE_DIR))
        actual_size = staged.stat().st_size
        if exp_size and actual_size != exp_size:
            staged.unlink(missing_ok=True)
            raise RuntimeError(f"Size mismatch for {fn}: got {actual_size}, expected {exp_size}")

        mlog(f"[{idx}/{len(items)}] Moving {actual_size/1e9:.2f} GB from local SSD -> Drive: {target}")
        partial = target.with_name(target.name + ".partial")
        h = hashlib.sha256()
        copied, last_log = 0, time.monotonic()
        with staged.open("rb") as src, partial.open("wb") as dst:
            while chunk := src.read(16 * 1024 * 1024):
                dst.write(chunk)
                if exp_sha:
                    h.update(chunk)
                copied += len(chunk)
                if time.monotonic() - last_log >= 20:
                    mlog(f"  copy progress {target.name}: {copied/1e9:.2f}/{(exp_size or actual_size)/1e9:.2f} GB")
                    last_log = time.monotonic()
        if exp_sha and h.hexdigest() != exp_sha:
            partial.unlink(missing_ok=True)
            staged.unlink(missing_ok=True)
            raise RuntimeError(f"SHA256 mismatch for {fn}: got {h.hexdigest()}, expected {exp_sha}")
        partial.replace(target)
        staged.unlink(missing_ok=True)
        dt = time.monotonic() - t0
        mlog(f"[{idx}/{len(items)}] DONE in {dt:.1f}s ({actual_size/1e6/max(dt,1):.1f} MB/s): {dest_rel}")
        manifest["files"][dest_rel] = {"repo": repo, "filename": fn, "size": actual_size, "sha256": exp_sha,
                                        "updated_at": time.strftime("%FT%T%z")}
        save_manifest(manifest)
        summary["completed"] += 1
        summary["files"].append({"dest": dest_rel, "size": actual_size, "status": "downloaded"})
        write_status(summary)
    else:
        url = it["url"]
        if target.is_file() and target.stat().st_size > 0:
            mlog(f"[{idx}/{len(items)}] SKIP (already on Drive): {dest_rel}")
            summary["skipped"] += 1
            summary["completed"] += 1
            summary["files"].append({"dest": dest_rel, "size": target.stat().st_size, "status": "cached"})
            write_status(summary)
            continue
        summary["current"] = {"index": idx, "dest": dest_rel, "source": url}
        write_status(summary)
        mlog(f"[{idx}/{len(items)}] FETCHING {url} -> {dest_rel}")
        partial = target.with_name(target.name + ".partial")
        urllib.request.urlretrieve(url, partial)
        partial.replace(target)
        sz = target.stat().st_size
        manifest["files"][dest_rel] = {"url": url, "size": sz, "updated_at": time.strftime("%FT%T%z")}
        save_manifest(manifest)
        summary["completed"] += 1
        summary["files"].append({"dest": dest_rel, "size": sz, "status": "downloaded"})
        write_status(summary)

summary["current"] = None
summary["finished_at"] = time.strftime("%FT%T%z")
write_status(summary)
shutil.copyfile(STATUS_PATH, COMFY_DATA / "models" / ".last-download-status.json")
mlog(f"All {summary['total']} items ready on Drive ({summary['skipped']} cached, {summary['total'] - summary['skipped']} newly downloaded).")


In [ ]:
# @title 2. Service -- lightweight status & manifest HTTP server on 127.0.0.1:8189
def start_service():
    subprocess.run(["pkill", "-f", f"http.server {SERVICE_PORT}"], check=False)
    return ct_spawn("service", [sys.executable, "-m", "http.server", str(SERVICE_PORT), "--bind", "127.0.0.1"],
                    cwd=WORK)

PROCS["service"] = (start_service(), start_service)
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/status.json", timeout=60)
print(f"comfyui-models status server listening on 127.0.0.1:{SERVICE_PORT}")


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        urllib.request.urlretrieve(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                   f"caddy_{tag.lstrip('v')}_linux_amd64.tar.gz", tgz)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={API_TOKEN}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        basic_auth {{
            {AUTH_USER} {hashed}
        }}
        header +Set-Cookie "ct_session={API_TOKEN}; Path=/; HttpOnly; SameSite=Lax"
        {upstream}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy; anything else means something else answers on that port.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Cloudflare tunnel -- token: fixed domain (named tunnel, created by the service script via `cf`)
#                               no token: temporary *.trycloudflare.com URL
CLOUDFLARED = ct_fetch_binary("/usr/local/bin/cloudflared",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")

def start_cloudflared():
    subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
    open(f"{WORK}/cloudflared.log", "w").close()
    if TUNNEL_TOKEN:   # ingress (hostname -> localhost:PUBLIC_PORT) lives in the Cloudflare config
        args = [CLOUDFLARED, "tunnel", "--no-autoupdate", "--protocol", "http2", "run", "--token", TUNNEL_TOKEN]
    else:
        args = [CLOUDFLARED, "tunnel", "--no-autoupdate", "--protocol", "http2",
                "--url", f"http://127.0.0.1:{PUBLIC_PORT}"]
    return ct_spawn("cloudflared", args)

def wait_public_url(timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        log = ct_tail("cloudflared", 400)
        if TUNNEL_TOKEN and "Registered tunnel connection" in log:
            return f"https://{PUBLIC_HOST}"
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if not TUNNEL_TOKEN and m:
            return m.group(0)
        time.sleep(1)
    raise RuntimeError("cloudflared did not come up:\n" + ct_tail("cloudflared"))

# A failing cell does not stop `colab exec`, so guard explicitly: no auth layer -> no tunnel.
if not globals().get("AUTH_OK"):
    raise RuntimeError("auth layer not ready (cell 3 failed) -- refusing to expose the service")
PROCS["cloudflared"] = (start_cloudflared(), start_cloudflared)
PUBLIC_URL = wait_public_url()
print(f"CT_PUBLIC_URL={PUBLIC_URL}")


In [ ]:
# @title 5. Ready + keep-alive -- KEEP THIS CELL RUNNING
# The Colab session only lives while the `colab exec` connection is open and this cell blocks.
# The watchdog restarts any process from PROCS that died; a heartbeat line is printed every 10 min.
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/status.json", timeout=3600)
if AUTH == "caddy" and not CREDS_FROM_ENV:   # started from the Colab UI: show generated credentials
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

last_beat = time.time()
while True:
    time.sleep(30)
    for name, (proc, restart) in list(PROCS.items()):
        if proc.poll() is not None:
            print(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 15)}", flush=True)
            PROCS[name] = (restart(), restart)
    if time.time() - last_beat > 600:
        last_beat = time.time()
        print(f"[heartbeat] {time.strftime('%H:%M:%S')} " +
              " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items()), flush=True)
